# General preprocessing
The cell below handles all the steps for preprocessing. The result is used as a starting point for both the subset prediction as the ChadsVasc score extraction. Below is another small cell that handles the training of a word2vec model, which is used is the LAAT model (subset prediction).

In [ ]:
from typing import List
import os
import re
import pandas as pd
from rapidfuzz import process, fuzz
import unicodedata
import re
import os
import json
from itertools import chain
from datetime import datetime

In [ ]:
FILE = r"add path to csv file"

df = pd.read_csv(FILE, delimiter="$")
print(len(df))
print(df.columns)

# Gathering of meta data (used for reporting in thesis)
In the following cells, we try to find out from which time span the notes date. 
Furthermore, we check how many entries have all zeros for every possible ICD-10 code. 

In [ ]:
for text in df["volledig_verslag_report_content"].head(5):
    print(text[:200])
    print("\n" + "-" * 80 + "\n")

In [ ]:
df["clean_text"] = df["volledig_verslag_report_content"].str.replace(
    r"\s+", " ", regex=True
)
for text in df["clean_text"].head(5):
    print(text[:200])
    print("\n" + "-" * 80 + "\n")

In [ ]:
pattern = r"van\s*(\d{2}/\s*\d{2}/\s*\d{4})\s*(?:t\.?e\.?m\.?|tot)\s*(\d{2}/\s*\d{2}/\s*\d{4})"

df[["opname_datum", "ontslag_datum"]] = df["clean_text"].str.extract(
    pattern, flags=re.IGNORECASE
)

df["ontslag_datum"] = df["ontslag_datum"].str.replace(" ", "")
df["opname_datum"] = df["opname_datum"].str.replace(" ", "")

df["opname_datum"] = pd.to_datetime(df["opname_datum"], dayfirst=True, errors="coerce")
df["ontslag_datum"] = pd.to_datetime(
    df["ontslag_datum"], dayfirst=True, errors="coerce"
)


df[["opname_datum", "ontslag_datum"]].head()

lowest_opname = df["opname_datum"].min()
highest_ontslag = df["ontslag_datum"].max()

print("Earliest admission:", lowest_opname)
print("Latest discharge:", highest_ontslag)

In [ ]:
row = df.loc[df["ontslag_datum"].idxmax()]

print("Ontslag datum:", row["ontslag_datum"])
print("\nReport text:\n")
print(row["clean_text"])

In [ ]:
# Get all diagnose columns
diagnose_cols = [col for col in df.columns if col.startswith("diagnose_")]

# Check rows where ALL diagnose columns are 0
all_zero_mask = (df[diagnose_cols] == 0).all(axis=1)

# View those rows
df[all_zero_mask]

# Or just get the count
print(f"Rows with all diagnose columns = 0: {all_zero_mask.sum()}")

# Preprocessing of raw data

In [ ]:
df = pd.read_csv(FILE, delimiter="$")
print(len(df))

dataset_name = "new_test_set_with_sections_ids"
FILE = f"./datasets/{dataset_name}.feather"
val_df = pd.read_feather(FILE)
print(len(val_df))

# test_df = pd.read_feather(FILE)
# print(len(test_df))

In [ ]:
def normalize_whitespace(text: str) -> str:
    text = re.sub(r"\n{3,}", "\n", text)
    text = re.sub(r"[ \t]{2,}", " ", text)
    return text.strip()


def normalize_line_endings(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    return text


def clean_note(text):
    pattern = r"""
        Patiënt\s[^\n]{0,15}\[LASTNAME\].*?
        Gegenereerd\ op\s+\d{2}/\d{2}/\d{4}\s+\d{2}:\d{2}\s+door\s+GEMS
    """
    text = re.sub(pattern, "", text, flags=re.VERBOSE | re.DOTALL)
    text = normalize_line_endings(text)
    return normalize_whitespace(text)


notes_column = "volledig_verslag_report_content"
df = df[
    df[notes_column].notna() & df[notes_column].astype(str).str.strip().astype(bool)
]
df["cleaned_note"] = df[notes_column].astype(str).apply(clean_note)
notes_column = "cleaned_note"


def merge_spaced_letters(title: str) -> str:
    # If most characters are single letters separated by spaces, merge them
    t_nospace = title.replace(" ", "")
    num_letters = len(t_nospace)
    num_spaces = title.count(" ")
    if num_letters == 0:
        return title
    # Heuristic: if more than 50% of characters are spaces → merge
    if num_spaces / num_letters > 0.5:
        return t_nospace
    return title


# -----------------------------------------------------
# 2. Uppercase TITLE extraction
# In this step, we also remove the notes who have less than 5 titles.
# This was done after inspecting the entries and checked with a clinical expert that those notes are less relevant for the problem statement.
# -----------------------------------------------------
print("-------------------------------")
print("Step 2: Uppercase title extraction")
title_pattern = re.compile(r"^\s*([A-Z][A-Z\s\-:]*?)\s*-*\s*$", re.MULTILINE)


def extract_uppercase_titles(text: str) -> List[str]:
    """Return all uppercase section titles in the text."""
    if not isinstance(text, str):
        return []
    matches = [line.strip() for line in title_pattern.findall(text) if line.strip()]
    cleaned_matches = [
        re.sub(r"[-:]+$", "", t).strip()  # remove trailing - or :
        for t in matches
        if t.strip()
    ]
    merged_titles = [merge_spaced_letters(t) for t in cleaned_matches]

    return merged_titles


df["uppercase_titles"] = df[notes_column].apply(extract_uppercase_titles)
TITLE_PATH = "titles_output"
os.makedirs(TITLE_PATH, exist_ok=True)

all_titles = sorted({title for titles in df["uppercase_titles"] for title in titles})
print(len(all_titles))

with open(os.path.join(TITLE_PATH, "uppercase_titles.txt"), "w", encoding="utf-8") as f:
    for title in all_titles:
        f.write(f"{title}\n")


df["num_titles"] = df["uppercase_titles"].apply(len)
notes_few_titles = df[df["num_titles"] < 5]
print(f"Removing notes with less than 5 titles, amount: {len(notes_few_titles)}")
print("-------------------------------")

# Print first 10 for inspection
# for i, row in notes_few_titles_diag.head(15).iterrows():
#     print(f"\n--- NOTE {i} ---")
# print(f"Number of titles: {row['num_titles']}")
# print("Titles extracted:", row["uppercase_titles"])
# print("Diagnosis columns with 1:")
# print({col: row[col] for col in diagnose_cols if row[col] == 1})
# print("Note text preview:", row[notes_column][:500])

# We remove the notes that have less than 5 titles
df = df[df["num_titles"] >= 5].copy()


# -----------------------------------------------------
# 3. Clustering of the titles
# -----------------------------------------------------
def normalize_title(title: str) -> str:
    title = title.upper().strip()
    title = re.sub(r"[-:]+$", "", title)
    title = "".join(
        c
        for c in unicodedata.normalize("NFD", title)
        if unicodedata.category(c) != "Mn"
    )
    title = re.sub(r"\s+", " ", title)
    return title


def cluster_titles(titles, threshold=80):
    """
    Group similar titles together using fuzzy matching.
    Returns a dict: {canonical_title: [variants]}
    """
    canonical = []

    clusters = {}

    for t in titles:
        t_norm = normalize_title(t)
        if not canonical:
            canonical.append(t_norm)
            clusters[t_norm] = [t]
            continue

        match, score, _ = process.extractOne(t_norm, canonical, scorer=fuzz.ratio)
        if score >= threshold:
            clusters[match].append(t)  # assign to existing cluster
        else:
            canonical.append(t_norm)
            clusters[t_norm] = [t]

    return clusters


clusters = cluster_titles(all_titles, threshold=80)
print(f"Total amount of titles after clustering: {len(clusters)}")
TITLE_PATH = "titles_output"
os.makedirs(TITLE_PATH, exist_ok=True)
output_file = os.path.join(TITLE_PATH, "clusters.json")
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(clusters, f, ensure_ascii=False, indent=4)
print(f"Clusters saved.")

In [ ]:
def canonical_title_dynamic(title: str) -> str:
    if any(word in title for word in ["GESTOPT", "STOP", "ONDERBROKEN"]):
        return title

    if title.startswith("ECHO") or title.startswith("ECHOGRAFIE"):
        return "ECHO"
    if title.startswith("CT"):
        return "CT"
    if title.startswith("DUPLEX"):
        return "DUPLEX"
    if title.startswith("RX"):
        return "RX"

    discharge_medication_keywords = [
        "MEDICATIE BIJ ONTSLAG",
        "THERAPIE BIJ ONTSAG",
        "ONTSLAGMEDICATIE",
        "ONTSLAGTHERAPIE",
        "MEDICATIELIJST BIJ ONTSLAG",
        "MEDICATIE BIJ ONTSLAG ITE",
        "MEDICATIE BIJ OTNSLAG",
        "MEDICATIE BIJ ONTSLAG",
        "MEDICATIE BIJ TRANSFER",
        "THERAPIE BIJ ONTSLAG: ONGEWIJZIGD",
        "THERAPIE BIJ ONTSLAG: ONVERANDERD",
        "THERAPIE BIJ TRANFER",
        "THERAPIE BIJ OVERDRACHT",
    ]
    if any(k in title for k in discharge_medication_keywords) or (
        "THERAPIE BIJ OVERNAME" in title
    ):
        return "ONTSLAG_MEDICATIE"

    medication_keywords = [
        "MEDICATIE",
        "MEDICATION",
        "MEDIKATIE",
        "THUISMEDICATI",
        "MEDICAMENTS",
    ]
    if any(k in title for k in medication_keywords) or (
        ("MEDICATIE" in title or "THERAPIE" in title) and "GESTOPT" not in title
    ):
        return "MEDICATIE"

    besluit_keywords = [
        "BSLUIT",
        "BESPREKING",
        "BEPREKING EN BESLUIT",
        "BELEID",
        "BELEID BIJ ONTSLAG",
        "CONCLUSIE",
        "BESLUIT",
        "BELSUIT",
    ]
    if any(k in title for k in besluit_keywords):
        return "BESLUIT"

    risk_keywords = [
        "CARDIALE RISICOFACTORE",
        "CARDIALE VOORGESCHIEDENIS",
        "CARDIOVASCULAIR RISK PROFIEL",
        "CARDIOVASCULAIR",
        "CARDIAAL RISICOPROFIEL",
        "CARDIALE ANTECEDENTEN",
        "RISICOPROFIEL",
    ]
    if any(k in title for k in risk_keywords) or "RISICO" in title or "RISK" in title:
        return "RISICOFACTOREN"

    antecedenten = ["MEDISCHE ANTCEDENTEN", "CARDIALE ANTECEDENTEN"]
    if any(k in title for k in antecedenten):
        return "ANTECEDENTEN"

    anamnese = [
        "ANAMESE",
        "ANAMNESE EN KLINISCH ONDERZOEK",
        "REDEN VAN CONSULT",
        "REDEN VAN DE OPNAME",
    ]
    if any(k in title for k in anamnese):
        return "ANAMNESE"

    return title


dynamic_clusters = {}
for canonical, variants in clusters.items():
    new_canonical = canonical_title_dynamic(canonical)

    if new_canonical not in dynamic_clusters:
        dynamic_clusters[new_canonical] = []

    dynamic_clusters[new_canonical].extend(variants)


print(f"\nNumber of canonical titles after merging: {len(dynamic_clusters)}")

# Path to save the JSON
output_path = r".\titles_output\final_clusters.json"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(dynamic_clusters, f, ensure_ascii=False, indent=4)
print(f"Final clusters saved to {output_path}")

In [ ]:
# -----------------------------------------------------
# 4. Split notes in sections
# -----------------------------------------------------
def extract_sections(note: str, variant_map: dict) -> dict:
    """
    Extract sections from a note and merge content for duplicate section titles.

    Returns a dict: {canonical_title: merged_content}
    """
    if not isinstance(note, str):
        return {}

    sections_dict = {}

    matches = list(title_pattern.finditer(note))

    for i, match in enumerate(matches):
        # Normalize and map to canonical title
        title = merge_spaced_letters(re.sub(r"[-:]+$", "", match.group(1).strip()))

        canonical = variant_map.get(title, title)

        start = match.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(note)
        content = note[start:end].strip()

        if not content:
            continue

        if canonical in sections_dict:
            sections_dict[canonical] += "\n" + content
        else:
            sections_dict[canonical] = content

    return sections_dict


RELEVANT_SECTIONS = {
    "ANAMNESE",
    "MEDICATIE",
    "RISICOFACTOREN",
    "ANTECEDENTEN",
    "BESLUIT",
}


def find_lowercase_section(note: str, section_name: str) -> str | None:
    """
    Try to detect a lowercase or mixed-case version of a section header.
    Returns the matched header line or None.
    """
    pattern = re.compile(
        rf"^\s*(?:[A-Z][a-z]+\s+)?{section_name.title()}[\s:]*$", re.MULTILINE
    )

    match = pattern.search(note)
    if match:
        return match.group(0).strip()

    return None


def extract_sections_with_fallback(note: str, variant_map: dict) -> dict:
    sections = extract_sections(note, variant_map)

    for section in RELEVANT_SECTIONS:
        if section not in sections:
            fallback = find_lowercase_section(note, section)
            if fallback:
                print(f"Fallback found for section '{section}': '{fallback}'")
                # Extract content starting from fallback header
                start = note.find(fallback) + len(fallback)
                content = note[start:].strip()

                if content:
                    sections[section] = content

    return sections


variant_to_canonical = {}
for canonical, variants in dynamic_clusters.items():
    for v in variants:
        variant_to_canonical[v.upper()] = canonical


df = val_df.copy()
df.drop(columns=["sections"])
notes_column = "text"

df["sections"] = df[notes_column].apply(
    lambda x: extract_sections_with_fallback(x, variant_to_canonical)
)
print(df["sections"].head(5))


def has_no_relevant_sections(section_dict):
    return not any(sec in RELEVANT_SECTIONS for sec in section_dict.keys())


mask_none_relevant = df["sections"].apply(has_no_relevant_sections)

count_none = mask_none_relevant.sum()

print(f"Number of entries with NONE of the relevant sections: {count_none}")


# Collect all section names from those rows
other_sections = df.loc[mask_none_relevant, "sections"].apply(lambda d: list(d.keys()))

flat_sections = list(chain.from_iterable(other_sections))

# Unique + sorted
unique_sections = sorted(set(flat_sections))

print("Section names found in entries WITHOUT relevant sections:")
for sec in unique_sections:
    print(sec)

# Keep only rows that have at least one relevant section
df = df[~mask_none_relevant].copy()

print(f"Remaining entries after removing notes with no relevant sections: {len(df)}")
if "text" in df.columns and "cleaned_note" in df.columns:
    df.drop(columns=["text"], inplace=True)  # remove the original
    df.rename(columns={"cleaned_note": "text"}, inplace=True)
    # df.drop(columns=['volledig_verslag_report_content'], inplace=True)


TITLE_PATH = "datasets"
os.makedirs(TITLE_PATH, exist_ok=True)

output_file = os.path.join(
    TITLE_PATH, f"{dataset_name}_{datetime.today().strftime('%d%m')}.feather"
)
df.to_feather(output_file)
print(f"DataFrame saved as Feather file at: {output_file}")

In [ ]:
df.to_feather(output_file)

# Repeat the process for the validation and test set
Here we use the titles and clusters that we found in the training set. 

In [ ]:
# 1. Clean notes
test_df = test_df[
    test_df["text"].notna() & test_df["text"].astype(str).str.strip().astype(bool)
]
test_df["cleaned_note"] = test_df["text"].astype(str).apply(clean_note)
notes_column = "cleaned_note"

# 2. Extract sections using the training set clusters
test_df["sections"] = test_df[notes_column].apply(
    lambda x: extract_sections_with_fallback(x, variant_to_canonical)
)

# 3. Remove notes with no relevant sections
mask_none_relevant = test_df["sections"].apply(has_no_relevant_sections)
print(f"Validation notes removed (no relevant sections): {mask_none_relevant.sum()}")
test_df = test_df[~mask_none_relevant].copy()

# 4. Handle column duplicates if needed
if "text" in test_df.columns and "cleaned_note" in test_df.columns:
    test_df.drop(columns=["text"], inplace=True)
test_df.rename(columns={"cleaned_note": "text"}, inplace=True)

# 5. Save to Feather
TITLE_PATH = "datasets"
os.makedirs(TITLE_PATH, exist_ok=True)
output_file = os.path.join(TITLE_PATH, "new_test_set_with_sections.feather")
test_df.to_feather(output_file)
print(f"Validation DataFrame saved at {output_file}")

# Training of word2Vec model on Jessa corpus
Here we train a word2vec model, so we can compare the performance with a general Dutch word2vec model.

In [ ]:
from gensim.models import Word2Vec
import re


## Tokenization ##
def simple_tokenize(text):
    text = text.lower()
    tokens = re.findall(r"\b\w+\b", text)
    return tokens


tokenized_texts = [simple_tokenize(t) for t in df["text"].tolist()]
w2v_model = Word2Vec(
    sentences=tokenized_texts, vector_size=200, window=5, min_count=2, workers=8
)
w2v_model.save("word2vec/jessa_w2v.model")


w2v_model.wv.save_word2vec_format("word2vec/jessa_w2v.bin", binary=True)
print("Saved Word2Vec .bin file: jessa_w2v.bin")